# 11 - Production and Serving
## Shakespeare Evidence Desk

- **Goal:** package the selected per-stage implementations as one composed pipeline behind a tested Application Programming Interface (API), refuse to ship any build that fails the stage 10 release gates, and choose the operating settings — caching and concurrency — that serve it within a latency and cost budget.
- **Experiment axis:** operating policy — answer **caching** on vs. off and **sequential** vs. **concurrent** request handling (the pipeline's answers are held fixed; only *how the service runs them* changes).
- **Loads from Steps 05-10:** the composed `production/pipeline.py` built from every stage handoff, plus `.artifacts/evaluation-results.json` (the gates and the promotion/rollback decision).
- **Exports:** `.artifacts/deployment-config.json` — the operating settings, the service contract (`/ask`, `/retrieve_evidence`, `/healthz`), and a release record stage 12 operates and monitors.


## Why This Stage

*In general — what production and serving is and why it is a distinct discipline from building the pipeline.* Everything through stage 10 answered "what is the best Retrieval-Augmented Generation (RAG) pipeline, and is it good enough?" This stage answers a different question: "how do we run it for real users, reliably and affordably, without ever shipping a build that failed its gates?" The intelligence is already decided; what remains is composition, enforcement, and operating settings — the unglamorous work that turns a notebook that answers one question into a service that answers thousands.

The core move is *composition without reinvention*. The selected configuration of every stage is assembled into one `production/pipeline.py` that the service calls, so there is exactly one implementation and it reflects the experiments, not kernel state. On top of that sits the gate check: the service reads stage 10's promotion decision and refuses to serve a build that did not pass, which is what makes continuous evaluation more than a report. The API itself is deliberately thin — a grounded answer, the evidence behind it, and a health probe — because the hard-won behavior lives in the pipeline, not the web layer.

The choice here is *measured*, like every stage, but the axis is operational. Serving the same answers can be fast and cheap or slow and expensive depending on two levers: whether we cache repeated work and whether we handle requests concurrently. We measure both against the questions we already have — warm versus cold latency, throughput as workers increase — and pick the simplest settings that meet a latency and cost budget without exceeding provider limits. The output is not a better answer; it is a deployable, gated service and the operating record stage 12 runs and watches.


## Conceptual Foundations

### From notebooks to a service: composition, not reinvention

Every stage so far ran its logic inside a notebook, rebuilt from the handoffs. Production **composes** those same selections into one callable path — `production/pipeline.py` — so the service does exactly what the experiments chose, with no notebook state and no second implementation to drift. The online flow is a single function: *query, retrieve, rerank, build context, generate*, returning a grounded, cited answer with a request id. The offline flow builds the index. The API is a thin shell over the online flow; the intelligence lives in the composed pipeline, not the web layer.

### The release gate: do not ship a build that fails evaluation

Stage 10 produced gates and a promotion/rollback decision. Production's first job is to **honor** them: before serving, the service checks the evaluation decision and refuses to start if the build did not pass. This is the whole point of continuous evaluation — the gate is not advisory. A build that regressed faithfulness, broke abstention, or dropped recall never reaches users; the last promoted build keeps serving instead.

### Latency has two regimes: cold and warm

A request's cost depends on whether its work was done before. A **cold** request computes everything — embeddings, retrieval, and a fresh generation call to the model (the expensive part). A **warm** request whose (query, context) was already answered serves from the cache in milliseconds. The gap between the two is enormous because a model generation dominates cold latency, so caching is the single largest serving lever. The on-disk cache the notebooks share *is* the production cache, which is why a warm service is fast and cheap.

### Throughput, concurrency, and an input/output-bound workload

**Latency** is how long one request takes; **throughput** is how many requests per second the service sustains. Because most of a cold request is spent *waiting* on the model provider, the workload is **input/output-bound**, and handling requests concurrently — with a worker pool — raises throughput well before any central-processing-unit limit bites. The gain is not unlimited: provider rate limits and the pool size cap it, and past a point adding workers stops helping.

### The service contract: `/ask`, `/retrieve_evidence`, `/healthz`

The API exposes the pipeline as a small, stable contract. `/ask` returns a grounded answer with citations, an abstention flag, and a request id. `/retrieve_evidence` returns just the cited passages, so another application (or a Large Language Model (LLM) tool call) can ground on the same authorized evidence without the generation step. `/healthz` is the liveness probe. Request ids make every answer traceable back through the telemetry stage 12 monitors.

### What production and serving is *not* (clean stage boundaries)

- **Not a pipeline stage:** it changes no retrieval or generation logic; it composes and serves the selections the earlier stages made.
- **Not evaluation (10):** it enforces the gates rather than computing them; it will not ship a failing build but it does not re-judge quality.
- **Not operations (12):** it stands the service up and picks operating settings; live monitoring, adversarial drills, alerting, and rollback execution are stage 12.

### Common misconceptions and failure modes

- **"Serving is just wrapping the pipeline in a web framework."** The web layer is trivial; the real work is composing the exact selections, honoring the gate, sharing the cache, and handling provider failures — none of which the framework does for you.
- **"Caching is free performance."** Caching is correct only while its keys are valid. Change a prompt, a model, or the corpus and stale entries must be invalidated, or the service serves yesterday's answer with today's confidence.
- **"More workers mean more speed."** Concurrency helps an input/output-bound service only until the provider rate limit or the pool saturates; beyond that, added workers add queueing, not throughput.
- **"A healthy process means a ready service."** Liveness is not readiness. The engine must load the index and warm its structures before the first real request, or that request pays the cold-start cost and may time out.


## The Technique Landscape

We **fix** the composed pipeline — every stage's selected configuration, already assembled in `production/pipeline.py` — and vary only the **operating policy** that wraps it. Two policies are measured head-to-head; the rest are explained so the choice is informed.

| Operating lever | Role here | What it does | Strengths | Weaknesses |
|---|---|---|---|---|
| **Answer caching** | compared | serve a repeated (query, context) from the on-disk cache instead of re-calling the model | collapses warm latency and cost to near zero | stale if the corpus or prompts change; adds a cache-invalidation concern |
| **Request concurrency** | compared | handle requests in parallel with a worker pool instead of one at a time | higher throughput on an input/output-bound workload | diminishing returns past a point; provider rate limits cap it |
| **Request batching** | explained, not run | coalesce several embeddings or generations into one provider call | fewer round trips, better token economics | adds latency while a batch fills; helps throughput, not single-request latency |
| **Streaming responses** | explained, not run | stream tokens to the client as they are generated | lower time-to-first-token for long answers | complicates citation validation and the abstention contract |
| **Horizontal scaling / autoscaling** | explained, not run | run many service replicas behind a load balancer | scales past a single host | an infrastructure concern beyond a single-service notebook |
| **Update / freshness policy** | explained, not run | how often the index and caches are rebuilt as the corpus changes | keeps answers current | a scheduling and invalidation concern, not a per-request setting |

So two per-request operating levers are measured; the rest frame the trade-off.


## 1. Load the Previous Selection

- **Purpose:** stand up the composed production pipeline and prove it is allowed to ship. Unlike earlier stages, this one does not rebuild the pipeline inline — it **imports** `production/pipeline.py`, the single composed path assembled from every stage handoff, which is the whole point of production: one implementation, driven by the selections.
- **What the code does:** locates the repository root and puts it on the path; imports the composed `answer`, `retrieve_evidence`, and `build_offline_index` from the production package; reads `.artifacts/evaluation-results.json` and **refuses to proceed** if stage 10's decision was rollback (a build that failed its gates must not be served); warms the engine so the first request pays no cold-start cost; and loads the labeled questions as the workload for the operating-policy benchmark.
- **Fails loudly, early:** if `evaluation-results.json` is missing, run notebook 10 first; if its decision is `rollback`, the notebook **stops here** — serving a failed build is the exact thing the gate exists to prevent. If the index is missing or was built on offline-fallback vectors, the engine warm-up raises.
- **Expected result:** the composed pipeline imported, the gate decision confirmed as promote, the engine warmed, and one sample grounded answer printed to show the end-to-end path works through the production module (served from the shared cache, so no live call on a warm run).


In [ ]:
import json
import sys
import time
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ARTIFACTS = ROOT / '.artifacts'

# Import the COMPOSED production pipeline - the single path assembled from every stage handoff.
from production.pipeline import answer, retrieve_evidence, build_offline_index, get_engine

# ---- honor the stage 10 release gate: do not serve a build that failed evaluation ----
evaluation_results_path = ARTIFACTS / 'evaluation-results.json'
if not evaluation_results_path.is_file():
    raise RuntimeError('Missing .artifacts/evaluation-results.json. Run notebook 10 first.')
evaluation_results = json.loads(evaluation_results_path.read_text(encoding='utf-8'))
gate_decision = evaluation_results.get('decision', 'rollback')
if gate_decision != 'promote':
    raise RuntimeError(
        f"Stage 10 decision is '{gate_decision}', not 'promote'. "
        "This build failed its release gates and must not be served. Fix the regression and re-run notebook 10.")

# Warm the engine once so the first real request does not pay the cold-start cost.
build_offline_index()
engine = get_engine()

questions_path = ARTIFACTS / 'eval-questions.json'
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}
workload = [q['question'] for q in evaluation_questions.get('development', []) + evaluation_questions.get('heldout', [])]
if not workload:
    raise RuntimeError('No labeled questions found to benchmark against. Run notebook 00 first.')

print(f'Composed pipeline imported from production/pipeline.py.')
print(f'Release gate decision: {gate_decision.upper()} - build is cleared to serve.')
print(f'Engine warmed: {len(engine.records)} indexed passages, policy={engine.params["policy"]}, '
      f'retrieval={engine.params["method"]}, contract={engine.params["contract"]}.')
print(f'Benchmark workload: {len(workload)} labeled question(s).')

sample = answer(workload[0])
print(f'\nSample /ask - "{workload[0][:60]}...":')
print(f'  abstained={sample["abstained"]}, citations={len(sample["citations"])}, latency={sample["latency_ms"]:.1f} ms')
print(f'  request_id={sample["request_id"]}')


## 2. Explain the Alternatives

An operating policy is **good** when it serves the gated answers within a latency and cost budget without exceeding provider limits or risking stale results. We **fix the pipeline** — the composed path and its answers are decided — and vary only *how the service runs it*.

### What we actually compare vs. what we only explain

| Approach | Role here | Why |
|---|---|---|
| **Answer caching (on vs. off)** | compared | serving a repeated (query, context) from disk instead of re-generating is the single largest latency and cost lever; measuring the gap quantifies what caching is worth |
| **Request concurrency (sequential vs. pooled)** | compared | the workload waits on the model provider, so a worker pool should raise throughput; measuring where the gain plateaus sets the worker count |
| **Request batching** | explained, not run | coalescing calls improves token economics and throughput but adds fill latency and does not help a single request; a throughput lever orthogonal to the two we tune |
| **Streaming responses** | explained, not run | lowers time-to-first-token but complicates citation validation and the abstention contract, which this service values more than perceived speed |
| **Horizontal scaling / autoscaling** | explained, not run | scales past one host but is infrastructure, not a per-request setting a notebook can measure |
| **Freshness / update policy** | explained, not run | governs when the index and caches are rebuilt; a scheduling and invalidation concern deferred to operations |

So two per-request levers are compared head-to-head; the rest are explained so the choice makes sense.

### How each compared lever actually works (a line or two)

- **Answer caching** — the online pipeline keys each generation by (deployment, contract, budget, context, question). A warm key returns the stored answer and its original latency in milliseconds; a cold key calls the model. Turning the cache off forces the cold path on every request. We measure warm latency directly and read the cold cost from the latencies the cache already recorded, so the comparison costs nothing to run.
- **Request concurrency** — the same warm workload is served first one request at a time, then through a thread pool of increasing size. Because each request is dominated by input/output wait, threads overlap that wait and raise throughput until the pool or the provider limit caps it.

### Why this is the experiment we run

The service must meet a budget, and only two per-request levers move the budget without touching the answers: caching (which collapses the dominant generation cost on repeats) and concurrency (which overlaps the unavoidable provider wait). Batching, streaming, and scaling are real but either orthogonal to single-request latency or beyond a single-service notebook, so they are explained, not measured. The cache key, the worker count, and the request timeout are bounded settings we choose from the measurements, not competing arms.


## 3. Run Controlled Experiments

- **Fixed pipeline:** every call goes through the imported `production.pipeline.answer`; the answers do not change between arms, only the serving policy around them.
- **What the code does:** benchmarks the **caching** lever by timing warm calls (`use_cache=True`, served from disk) over the workload and reading the **cold** generation cost from the latencies the generation cache already recorded, so no live model call is billed; then benchmarks the **concurrency** lever by serving the same warm workload sequentially and through thread pools of increasing size, recording wall-clock throughput.
- **Cache-safe by design:** the warm path reads disk only, and the cold estimate is taken from previously recorded latencies rather than by forcing fresh generations. A warm run of this notebook therefore makes zero live calls and bills nothing, while still producing an honest cold-vs-warm comparison.
- **What is measured:** per-request latency (warm and cold-estimate), the median and 95th-percentile of each, and throughput in requests per second at each concurrency level.
- **Expected result:** the two registered levers, the warm and cold latency samples, the throughput-versus-workers curve, and confirmation that the benchmark ran entirely on the shared cache.


In [ ]:
from concurrent.futures import ThreadPoolExecutor


def percentile(values, fraction):
    if not values:
        return 0.0
    ordered = sorted(values)
    return ordered[min(len(ordered) - 1, int(fraction * len(ordered)))]


# ---- Lever 1: caching. Warm latency is measured; cold latency is read from the generation cache. ----
warm_latencies = []
for question in workload:
    started = time.perf_counter()
    answer(question, use_cache=True)   # disk-served on a warm cache
    warm_latencies.append((time.perf_counter() - started) * 1000)

# Cold cost = the real first-generation latencies the cache recorded, without re-billing the provider.
GEN_CACHE = ARTIFACTS / 'generation-cache'
cold_latencies = []
for cache_file in GEN_CACHE.glob('*.json'):
    try:
        recorded = json.loads(cache_file.read_text()).get('latency_ms', 0.0)
    except (json.JSONDecodeError, OSError):
        continue
    if recorded and recorded > 0:
        cold_latencies.append(recorded)

caching_rows = [
    ('cache off (cold)', percentile(cold_latencies, 0.5), percentile(cold_latencies, 0.95), len(cold_latencies)),
    ('cache on (warm)', percentile(warm_latencies, 0.5), percentile(warm_latencies, 0.95), len(warm_latencies)),
]

print('Caching lever (per-request latency, milliseconds):')
print('{:<18} {:>10} {:>10} {:>8}'.format('policy', 'p50 ms', 'p95 ms', 'n'))
for name, p50, p95, n in caching_rows:
    print('{:<18} {:>10.1f} {:>10.1f} {:>8}'.format(name, p50, p95, n))
speedup = (percentile(cold_latencies, 0.95) / percentile(warm_latencies, 0.95)) if percentile(warm_latencies, 0.95) else float('inf')
print(f'Warm cache is ~{speedup:.0f}x faster at the 95th percentile than the cold path.')


# ---- Lever 2: concurrency. Serve the warm workload sequentially vs through thread pools. ----
def throughput(worker_count, repeats=3):
    batch = workload * repeats
    started = time.perf_counter()
    if worker_count == 1:
        for question in batch:
            answer(question, use_cache=True)
    else:
        with ThreadPoolExecutor(max_workers=worker_count) as pool:
            list(pool.map(lambda q: answer(q, use_cache=True), batch))
    elapsed = time.perf_counter() - started
    return len(batch) / elapsed if elapsed else 0.0


concurrency_levels = [1, 2, 4, 8]
concurrency_throughput = {workers: throughput(workers) for workers in concurrency_levels}

print('\nConcurrency lever (warm workload throughput):')
print('{:<10} {:>14} {:>12}'.format('workers', 'req/sec', 'vs 1 worker'))
baseline_rps = concurrency_throughput[1] or 1e-9
for workers in concurrency_levels:
    rps = concurrency_throughput[workers]
    print('{:<10} {:>14.1f} {:>11.1f}x'.format(workers, rps, rps / baseline_rps))


## Metrics in This Stage — What Each Number Means

Section 4 compiles the operating measurements, picks the worker count where throughput stops improving, and verifies the Application Programming Interface (API) actually serves the contract. The selection keeps the simplest settings that meet the latency and cost budget.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Warm p95 latency | serving cost when the work was already done | 95th percentile of per-request time with the cache on | The latency users see on repeated or cached questions; lower is better. |
| Cold p95 latency | serving cost of a fresh generation | 95th percentile of the first-generation latencies the cache recorded | The worst-case latency on a cache miss; dominated by the model call. |
| Cache speedup | what caching is worth | cold p95 divided by warm p95 | Quantifies the single largest serving lever; higher means caching matters more. |
| Throughput at W workers | requests per second at concurrency W | warm workload size divided by wall-clock time with W workers | How many users the service sustains; higher is better until it plateaus. |
| Concurrency scaling | how well throughput grows with workers | throughput at W divided by throughput at one worker | Confirms the input/output-bound gain and where it saturates. |
| Warm-up time | cold-start cost paid once at boot | time to build the engine on startup | Why readiness must wait for warm-up; a one-time, not per-request, cost. |
| API contract check | whether the service actually serves | status codes and shapes from `/healthz` and `/ask` via a test client | Confirms the web layer wires the pipeline correctly; must pass. |


## 4. Measure and Inspect

- **The lenses:** **warm vs. cold latency** and the **cache speedup** (what caching is worth), the **throughput-versus-workers** curve and where it **plateaus** (the right concurrency), the one-time **warm-up** cost, and a live **API contract check** (does the service actually serve `/healthz` and `/ask`).
- **Caching dominates latency.** The warm path serves from disk in milliseconds while the cold path pays a full model generation, so the speedup is large and the decision — cache on — is effectively made by the numbers. The remaining caching concern is invalidation, not whether to cache.
- **Concurrency pays until it plateaus.** Because each request waits on the provider, throughput rises with workers, but the gain flattens once the pool saturates or the provider limit bites. We pick the smallest worker count at the knee of the curve rather than the maximum, because workers past the plateau add queueing, not speed.
- **Readiness is not liveness.** Warm-up builds the engine once; the service must finish it before accepting traffic. We measure that one-time cost so the deployment can set a readiness delay rather than letting the first user pay it.
- **The API is verified in-process.** Using a test client (no network, no running server) we call `/healthz` and `/ask` and check the status codes and response shape, confirming the web layer wires the composed pipeline correctly. If the serving dependency is not installed, this check is skipped with a clear message rather than failing the notebook.
- **Honest limit:** this is a single-process, warm-cache micro-benchmark on a handful of questions, not a load test against a deployed host. It sizes the operating settings and proves the contract; true capacity and tail latency under real traffic are stage 12's live concern.


In [ ]:
warm_p95 = percentile(warm_latencies, 0.95)
cold_p95 = percentile(cold_latencies, 0.95)
cache_speedup = (cold_p95 / warm_p95) if warm_p95 else float('inf')

# Pick the smallest worker count at the knee: stop when the next step adds < 10% throughput.
PLATEAU_GAIN = 0.10
selected_workers = concurrency_levels[0]
for previous, current in zip(concurrency_levels, concurrency_levels[1:]):
    gain = (concurrency_throughput[current] - concurrency_throughput[previous]) / (concurrency_throughput[previous] or 1e-9)
    selected_workers = previous
    if gain < PLATEAU_GAIN:
        break
else:
    selected_workers = concurrency_levels[-1]

# One-time warm-up cost: rebuild the engine from a cold cache (the lru_cache is already warm, so time a fresh build).
get_engine.cache_clear()
warmup_started = time.perf_counter()
build_offline_index()
warmup_ms = (time.perf_counter() - warmup_started) * 1000

print('Operating summary:')
print(f'  cache: warm p95={warm_p95:.1f} ms vs cold p95={cold_p95:.1f} ms  ->  ~{cache_speedup:.0f}x speedup')
print(f'  concurrency: throughput {concurrency_throughput[1]:.1f} -> {concurrency_throughput[selected_workers]:.1f} req/s '
      f'at {selected_workers} worker(s) (knee of the curve)')
print(f'  warm-up: {warmup_ms:.0f} ms one-time engine build at startup')

# ---- live API contract check via an in-process test client (no network, no running server) ----
api_contract_ok = False
try:
    from fastapi.testclient import TestClient

    from production.api.main import create_app

    test_client = TestClient(create_app())
    health = test_client.get('/healthz')
    ask = test_client.post('/ask', json={'query': workload[0]})
    api_contract_ok = (
        health.status_code == 200 and health.json().get('status') == 'ok'
        and ask.status_code == 200 and 'answer' in ask.json() and 'request_id' in ask.json()
    )
    print(f'\nAPI contract check: /healthz={health.status_code}, /ask={ask.status_code} -> '
          f'{"PASS" if api_contract_ok else "FAIL"}')
    print(f'  /ask returned {len(ask.json().get("citations", []))} citation(s), '
          f'abstained={ask.json().get("abstained")}')
except ImportError:
    print('\nAPI contract check skipped: install fastapi and httpx to verify the service in-process '
          '(pip install -r requirements.txt).')


## 5. Justify the Selection

- **What is being selected:** the operating settings — whether to cache, how many workers, and the request timeout — not any pipeline logic. The answers are fixed and gated; this is purely how the service runs them.
- **Decision rule (data-driven, deterministic):** enable answer caching, because the measured warm-versus-cold speedup is large and the only cost is invalidation discipline; set the worker count to the **knee** of the throughput curve (the smallest count past which an added worker yields less than ten percent more throughput), because workers beyond the knee add queueing rather than speed; and set the request timeout generously above the cold p95 so a legitimate cache-miss generation is not killed mid-flight.
- **Why caching is not a trade-off here:** a repeated question with identical context is the *same* answer; recomputing it spends a model call to reproduce a known result. The only reason not to cache is staleness, which is handled by invalidating on any change to the model, prompts, or corpus — a freshness policy, not an argument against caching.
- **Why the knee, not the maximum:** more workers help only while requests are waiting on the provider. Past the knee the pool and the provider rate limit cap throughput, so additional workers raise memory and queueing with no latency benefit. The simplest setting that captures the gain is the smallest worker count at the plateau.
- **Expected result:** the chosen operating settings printed with the measurements that justify them, ready to export as the deployment configuration.


In [ ]:
# Request timeout set above the cold p95 with headroom, so a legitimate cache-miss generation is not killed.
REQUEST_TIMEOUT_MS = max(30000, int(cold_p95 * 3))

operating_settings = {
    'answer_cache_enabled': True,
    'workers': selected_workers,
    'request_timeout_ms': REQUEST_TIMEOUT_MS,
    'max_query_chars': 2000,
    'retry_attempts': 3,
}

print('Selected operating settings:')
print(f"  answer cache: {'enabled' if operating_settings['answer_cache_enabled'] else 'disabled'} "
      f"(~{cache_speedup:.0f}x warm speedup)")
print(f"  workers: {operating_settings['workers']} (knee of the throughput curve)")
print(f"  request timeout: {operating_settings['request_timeout_ms']} ms (> cold p95 of {cold_p95:.0f} ms)")
print(f"  input guardrail: reject queries over {operating_settings['max_query_chars']} chars")
print(f"  transient retries: {operating_settings['retry_attempts']} with exponential backoff")
print('\nRejected alternatives:')
print('  cache off: pays a full model generation on every repeat - rejected on latency and cost')
print(f'  workers > {selected_workers}: past the throughput knee, adds queueing not speed - rejected on simplicity')


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/deployment-config.json` — the release record stage 12 operates: the operating settings (caching, workers, timeout, guardrail, retries), the service contract (`/ask`, `/retrieve_evidence`, `/healthz`), the measured latency and throughput, a reference to the stage 10 gate decision that cleared this build, and the composed pipeline's configuration fingerprint.
- **Why a deployment record, not a code dump:** the service *is* `production/pipeline.py` plus the API shell; the handoff captures how to *run* it — the settings, the contract, and the evidence that it passed its gates — so operations can stand it up, monitor it, and roll it back against a known-good baseline.
- **How it actually ships:** `docker-compose.yml` runs the service with `uvicorn production.api.main:create_app --factory`; the same package the notebook imported is what the container serves, so there is no second implementation to drift.
- **Production targets filled this stage:** `production/pipeline.py` (the composed online and offline flows), `production/api/main.py` and `production/api/routes.py` (the FastAPI service), and `production/tools/llm_tool.py` and `production/tools/mcp_server.py` (the reusable `retrieve_evidence` tool).
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff or the caches.
- **Expected result:** the handoff written, with the operating settings, the service contract, and the gate reference printed.


In [ ]:
deployment_config_path = ARTIFACTS / 'deployment-config.json'

# deployment-config.json: how to RUN the gated service - operating settings, contract, and the evidence it passed.
deployment_config = {
    'stage': '11-production-and-serving',
    'release_gate': {
        'source': 'evaluation-results.json',
        'decision': gate_decision,
        'release_judge': evaluation_results.get('release_judge'),
    },
    'operating_settings': operating_settings,
    'service_contract': {
        'framework': 'fastapi',
        'run_command': 'uvicorn production.api.main:create_app --factory --host 0.0.0.0 --port 8000',
        'endpoints': {
            'POST /ask': 'grounded answer with citations, abstention flag, and request id',
            'POST /retrieve_evidence': 'cited passages for a query, for tool use (no generation)',
            'GET /healthz': 'liveness probe',
        },
    },
    'measured': {
        'warm_p95_ms': round(warm_p95, 1),
        'cold_p95_ms': round(cold_p95, 1),
        'cache_speedup': round(cache_speedup, 1),
        'throughput_by_workers': {str(w): round(concurrency_throughput[w], 2) for w in concurrency_levels},
        'selected_workers': selected_workers,
        'warmup_ms': round(warmup_ms, 1),
        'api_contract_ok': api_contract_ok,
    },
    'pipeline_fingerprint': {
        'query_policy': engine.params['policy'],
        'retrieval_method': engine.params['method'],
        'reranker': engine.params['reranker'],
        'context_builder': engine.params['builder'],
        'grounding_contract': engine.params['contract'],
        'top_k': engine.params['top_k'],
    },
    'production_targets': [
        'production/pipeline.py', 'production/api/main.py', 'production/api/routes.py',
        'production/tools/llm_tool.py', 'production/tools/mcp_server.py',
    ],
}
deployment_config_path.write_text(json.dumps(deployment_config, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote deployment record to {deployment_config_path.relative_to(ROOT)}')
print(f'Serving (gate={gate_decision}): cache on, {selected_workers} worker(s), timeout {REQUEST_TIMEOUT_MS} ms.')
print('Run locally with: docker compose up  (or the uvicorn command in the service contract).')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit - do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **The service is the composed pipeline, and that is the point.** The hardest work here was not the web layer — it was composing every stage's selection into one path that shares the notebooks' caches and reflects the handoffs exactly. The API is deliberately thin so there is nothing to drift.
- **Caching made the benchmark both fast and honest.** Serving from the shared disk cache let the notebook measure warm latency and read the cold cost from recorded latencies without re-billing the provider. The same cache that makes the service cheap makes the experiment reproducible.
- **The concurrency number is a shape, not a capacity.** The throughput curve shows the input/output-bound gain and its knee on a single process with a warm cache. It picks a sensible worker count, but it is not a load test; real capacity depends on the host, the provider limit, and the live cold-miss rate.
- **Gate enforcement is a one-line idea with outsized value.** Refusing to serve a rolled-back build is a few lines, but it is what turns stage 10 from a report into a control. The risk is forgetting to re-run evaluation after a change, so the gate check reads the artifact rather than trusting memory.
- **Invalidation is the caching debt.** Caching is only correct while its keys are valid. The moment a prompt, model, or the corpus changes, stale entries must be cleared. That freshness policy is noted here and belongs to operations.
- **How we will revisit.** Add streaming and batching once the contract is stable, replace the micro-benchmark with a real load test, and wire the request ids into end-to-end traces so stage 12 can watch tail latency and errors on live traffic.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the production package and the notebook 10 artifact, not hidden state. The benchmark runs on the shared warm caches, so a clean run makes no live calls and bills nothing.
- **What we achieved:** the selected per-stage implementations composed into `production/pipeline.py`; the stage 10 release gate enforced before serving; two operating levers measured (caching and concurrency) and the simplest passing settings chosen; a tested FastAPI service exposing `/ask`, `/retrieve_evidence`, and `/healthz`, verified in-process; and an exported deployment configuration.
- **Honest limits:** this is a single-process, warm-cache micro-benchmark on a handful of questions, not a load test against a deployed host. It sizes the operating settings and proves the contract; real capacity and tail latency under production traffic are stage 12's live concern.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff or the caches.
- **Next stage — 12 Operations and Safety:** load `.artifacts/deployment-config.json`, then operate the service safely — monitor and alert on live traffic, run adversarial drills (prompt injection, data exfiltration, out-of-corpus abstention), handle provider failures and throttling with graceful degradation, and rehearse rollback to the last promoted build.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. What does "production" add if the pipeline already works in a notebook?**
Composition, enforcement, and operation. The notebook rebuilt the pipeline from handoffs every run; production assembles the selections into one `production/pipeline.py` the service calls, so there is a single implementation that cannot drift from the experiments. On top of that it enforces the stage 10 gate (a failed build will not serve) and chooses how to run the service affordably. None of that is about answer quality; it is about turning a correct pipeline into a dependable service.

**Q2. Why is honoring the release gate part of serving, not evaluation?**
Because a gate that nothing enforces is just a report. Stage 10 computes whether a build is good enough; stage 11 is the place that *acts* on it by refusing to start a rolled-back build. Putting the check at the serving boundary is what makes continuous evaluation load-bearing — the gate stands between a regression and real users.

**Q3. Why is caching the single biggest serving lever?**
Because a cold request is dominated by a model generation, and a warm request skips it entirely. Everything else in the request — embedding, retrieval, context assembly — is cheap by comparison. Caching the (query, context, answer) turns the expensive, variable part of the request into a millisecond disk read, which is why the warm-versus-cold gap is measured in multiples, not percentages.

**Q4. If more workers raise throughput, why not use as many as possible?**
Because the workload is input/output-bound, not compute-bound: workers help only while requests sit waiting on the provider. Once the pool is large enough to cover that wait, the provider's rate limit becomes the ceiling and extra workers just queue. The knee of the throughput curve is the point of diminishing returns, and the simplest setting that captures the gain is the smallest worker count at that knee.

**Q5. Why verify the API with a test client instead of a running server?**
An in-process test client exercises the exact application object — routing, validation, and the pipeline call — without binding a port or needing the service to be up, so it runs deterministically inside the notebook. It proves the web layer wires the composed pipeline correctly, which is all the contract check needs; true network behavior and capacity are a load-test concern for operations.

**Q6. The benchmark is tiny and warm — what does it actually establish?**
It sizes the two operating settings and proves the contract; it does not measure real capacity. Warm single-process timings on a handful of questions are enough to show caching dominates latency and to find the concurrency knee, but tail latency under real, mixed, partly-cold traffic is something only live monitoring can tell us. The handoff records this limit plainly.

**Q7. What did we produce, and what happens next?**
We composed `production/pipeline.py`, wrapped it in a tested FastAPI service with `/ask`, `/retrieve_evidence`, and `/healthz`, chose caching and concurrency from measurements, enforced the stage 10 gate, and exported `.artifacts/deployment-config.json`. Stage 12 (Operations and Safety) runs this service for real: monitoring and alerting on live traffic, adversarial drills for prompt injection and exfiltration, provider-failure and throttling handling, and rehearsed rollback.
